In [ ]:
import os
import pandas as pd
from huggingface_hub import InferenceClient
from transformers import AutoTokenizer

# Initialize client
client = InferenceClient(
    provider="hf-inference",
    api_key=""
)

# Load tokenizer for proper truncation (same model base)
tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")

In [ ]:
df = pd.read_csv('../data/dcInbox/dcinbox_export_114.csv')
unnamed_cols = df.columns.str.contains('^Unnamed')
df = df.loc[:, ~unnamed_cols].copy()
df = df[pd.to_numeric(df['Unix Timestamp'], errors='coerce').notna()].copy()
df['datetime'] = pd.to_datetime(df['Unix Timestamp'], unit='ms')

# Sort chronologically
df = df.sort_values('datetime').reset_index(drop=True)

# Filter data for February and March
# df = df[df['datetime'].between('2016-01-01', '2016-12-31')]
df = df[df['Chamber'] == 'House']
TEST_N = 10
df = df.head(TEST_N).copy() 
republican_emails = df[df['Party'] == 'Republican']
democrat_emails = df[df['Party'] == 'Democrat']

In [ ]:
import time
from tqdm import tqdm
import re

def truncate_text(text, max_tokens=510):
    """
    Truncate text to fit within the model's token limit.
    Uses the tokenizer to properly truncate while preserving token boundaries.
    Uses 510 tokens to leave room for special tokens that the API adds.
    """
    # Convert to string and handle None/NaN
    text = str(text) if pd.notna(text) else ""
    if not text:
        return None
    
    # Strip whitespace and clean up
    text = text.strip()
    if not text:
        return None
    
    # Remove null bytes and other problematic characters
    text = text.replace('\x00', '').replace('\ufffd', '')
    
    # Try tokenizer-based truncation
    try:
        # Tokenize the text (without special tokens, API will add them)
        tokens = tokenizer.encode(text, add_special_tokens=False, max_length=max_tokens, truncation=True)
        
        # Check if we have any tokens
        if len(tokens) == 0:
            return None
        
        # Decode back to text (this ensures proper truncation)
        truncated_text = tokenizer.decode(tokens, skip_special_tokens=True)
        
        # Clean up the text - remove extra whitespace
        truncated_text = re.sub(r'\s+', ' ', truncated_text).strip()
        
        # Ensure we have at least some meaningful content (at least 10 chars)
        if not truncated_text or len(truncated_text) < 10:
            # Fallback: try to get first 500 characters of original text
            fallback_text = text[:500].strip()
            if fallback_text and len(fallback_text) >= 10:
                return fallback_text
            return None
        
        return truncated_text
    except Exception as e:
        # Fallback to simple character-based truncation if tokenization fails
        fallback_text = text[:500].strip()
        if fallback_text and len(fallback_text) >= 10:
            return fallback_text
        return None

# Initialize empty list to store results
results = []

# Process each email
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing emails"):
    try:
        # Truncate email body to fit within token limit (510 tokens)
        email_body = truncate_text(row['Body'], max_tokens=510)
        
        # Skip if text is None or too short
        if email_body is None or len(email_body.strip()) < 10:
            # Add row with NaN for emotion scores
            row_data = {
                'date': row['datetime'],
                'first_name': row['First Name'],
                'last_name': row['Last Name'],
                'subject': row['Subject'],
                'id': row['ID'],
            }
            results.append(row_data)
            continue
        
        # Get emotion classification for the email body
        emotion_result = client.text_classification(
            email_body,
            model="j-hartmann/emotion-english-distilroberta-base",
        )
        
        # Create a dictionary for this row
        row_data = {
            'date': row['datetime'],
            'first_name': row['First Name'],
            'last_name': row['Last Name'],
            'subject': row['Subject'],
            'id': row['ID']
        }
        
        # Add each emotion score as a column
        for emotion in emotion_result:
            # Convert emotion label to column name (e.g., 'joy' -> 'joy_score')
            emotion_key = f"{emotion['label'].lower()}_score"
            row_data[emotion_key] = emotion['score']
        
        results.append(row_data)
        
        # Small delay to avoid rate limiting
        time.sleep(0.1)
        
    except Exception as e:
        print(f"Error processing row {idx}: {e}")
        # Still add the row with NaN for emotion scores
        row_data = {
            'date': row['datetime'],
            'first_name': row['First Name'],
            'last_name': row['Last Name'],
            'subject': row['Subject'],
            'id': row['ID']
        }
        results.append(row_data)

# Create dataframe from results
emotion_results_df = pd.DataFrame(results)
emotion_results_df.head()

In [ ]:
# import pandas as pd
# emotion_results_df = pd.read_csv('../data/emotion_scores/emotion_scores_dcinbox_114.csv')
# test = emotion_results_df[emotion_results_df['id'] == 117975]
# test